In [75]:
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("talesgomes27/sherleck-books")

print("Path to dataset files:", path)

dataset_path = 'the_adventures_of_sherlock_holmes.txt'

path = os.path.join(path, dataset_path)
with open(path) as f:
    text = f.read()

Path to dataset files: /Users/s932172@aics.espritscholen.nl/.cache/kagglehub/datasets/talesgomes27/sherleck-books/versions/3


In [76]:
import numpy as np

# text = """hello there people. how are you? I'm doing very well, and I am living happily. People these days are so dumb."""
text = str.lower(text)
text = text.replace('\n', ' ')


text = "hello there people. hello."

# version with words instead of chars (variable chars now represents words)

split = text.split()

chars = sorted(list(set(split)))

char_to_idx = {ch: i for i, ch in enumerate(chars)}

idx_to_char = {i: ch for i, ch in enumerate(chars)}

# print(char_to_idx)
# print(idx_to_char)

text_indices = [char_to_idx[ch] for ch in split]

# print(text_indices)

In [77]:
sequence_length = 2

X = []
y = []

for i in range(len(text_indices) - sequence_length):
    X.append(text_indices[i:i+sequence_length])

    y.append(text_indices[i+sequence_length])


X = np.array(X)
y = np.array(y)
# print(X)
# print(y)

In [78]:
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import SimpleRNN, Dense, Input


model = Sequential()
model.add(Input((X.shape[1],len(chars))))
model.add(SimpleRNN(128, activation='relu'))


model.add(Dense(len(chars), activation='softmax'))


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
metrics=['accuracy'])


In [79]:
X_onehot = np.zeros((X.shape[0], X.shape[1], len(chars)))

for i, sequence in enumerate(X):
    for t, char_idx in enumerate(sequence):
        X_onehot[i, t, char_idx] = 1.0

# print(X_onehot)

# y = np.expand_dims(y, axis=-1)

In [80]:
model.fit(X_onehot, y, epochs=30, batch_size=32)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.0000e+00 - loss: 1.5451
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0000e+00 - loss: 1.4781
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0000e+00 - loss: 1.4139
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0000e+00 - loss: 1.3541
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5000 - loss: 1.2963
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5000 - loss: 1.2400
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 1.1859
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 1.1340
Epoch 9/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 1.0832
Epoch 10/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 1.0331
Epoch 11/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.9850
Epoch 12/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000

In [81]:
def predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length):
    if len(seed) < sequence_length:
        seed = [""] * (sequence_length - len(seed)) + seed
    elif len(seed) > sequence_length:
        seed = seed[-sequence_length:]

    seed_indices = [char_to_idx.get(ch,0) for ch in seed]

    X_seed = np.zeros((1, sequence_length, len(char_to_idx)))
    for t, idx in enumerate(seed_indices):
        X_seed[0, t, idx] = 1.0

    prediction = model.predict(X_seed, verbose=0)

    predicted_idx = np.argmax(prediction, axis=-1)[0]

    predicted_char = idx_to_char[predicted_idx]

    return predicted_char

In [82]:
seed = ['hello','there']

# predicted_char = predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length)
# print(f"Predicted next character: {predicted_char}")
print(f"Text: {' '.join(seed)}",end=' ')

predicted_char = predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length)
print(predicted_char,end=' ')

# for i in range(20):
#     predicted_char = predict_next_char(model, seed, char_to_idx, idx_to_char, sequence_length)
#     print(predicted_char,end=' ')
#     seed+=predicted_char
    

Text: hello there people. 